# Khon mask dense reconstruction (Colab GPU)

COLMAP's `patch_match_stereo` requires CUDA. The project is developed on an
Apple Silicon Mac, where the local build reports *"without CUDA"* and dense
stereo cannot run at all. This notebook is the one stage that runs elsewhere.

**Before running:** `Runtime > Change runtime type > T4 GPU`.

Pipeline: upload the bundle from `scripts/02_dense_export.py`, undistort,
run patch-match stereo, fuse, download `fused.ply`.

Then back on your machine:
```
python scripts/03_dense_import.py ~/Downloads/fused.ply -s paths.run_id=<run>
```

## 1. Check the GPU

Stop here if this fails -- the rest cannot work without CUDA.

In [ ]:
import subprocess, sys

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode != 0:
    sys.exit(
        "No GPU detected. Set Runtime > Change runtime type > T4 GPU, then "
        "Runtime > Restart session and run this cell again."
    )
print(result.stdout.split("\n")[8] if len(result.stdout.split("\n")) > 8 else result.stdout)
print("GPU OK")

## 2. Install a CUDA-enabled COLMAP

`apt install colmap` is tried first because it is fast. Debian's build is
**not** guaranteed to have CUDA, so the next cell verifies rather than assumes;
if it lacks CUDA we fall back to a conda-forge build, which does.

Takes a few minutes on first run.

In [ ]:
%%bash
set -e
apt-get -qq update
DEBIAN_FRONTEND=noninteractive apt-get -qq install -y colmap > /dev/null 2>&1 || true
colmap --help 2>&1 | head -1 || echo 'colmap not installed via apt'

In [ ]:
import shutil, subprocess

def colmap_banner():
    if shutil.which("colmap") is None:
        return ""
    out = subprocess.run(["colmap", "--help"], capture_output=True, text=True)
    return ((out.stdout or "") + (out.stderr or "")).splitlines()[0]

banner = colmap_banner()
print("apt build:", banner or "(none)")

NEEDS_CONDA = (not banner) or ("without CUDA" in banner)
print("CUDA-capable:", not NEEDS_CONDA)
if NEEDS_CONDA:
    print("-> falling back to the conda-forge CUDA build in the next cell")

In [ ]:
if NEEDS_CONDA:
    !curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba > /dev/null 2>&1
    !./bin/micromamba create -y -p /content/mamba-colmap -c conda-forge \
        'colmap=*=*cuda*' > /dev/null 2>&1 || \
     ./bin/micromamba create -y -p /content/mamba-colmap -c conda-forge colmap > /dev/null 2>&1
    import os
    os.environ["PATH"] = "/content/mamba-colmap/bin:" + os.environ["PATH"]
    print(colmap_banner())

In [ ]:
banner = colmap_banner()
assert banner, "COLMAP is not installed"
assert "without CUDA" not in banner, (
    f"This COLMAP build has no CUDA support ({banner}). patch_match_stereo "
    "cannot run. Try restarting the runtime with a GPU enabled."
)
print("OK:", banner)

## 3. Upload the bundle

Upload `dense_bundle_<run_id>.zip`, produced by `scripts/02_dense_export.py`.

For bundles over ~200 MB, mounting Drive is more reliable than the upload
widget -- use the commented alternative.

In [ ]:
import json, zipfile, pathlib
from google.colab import files

WORK = pathlib.Path("/content/work")
WORK.mkdir(exist_ok=True)

uploaded = files.upload()
bundle = pathlib.Path(list(uploaded)[0])

# Alternative for large bundles:
# from google.colab import drive; drive.mount('/content/drive')
# bundle = pathlib.Path('/content/drive/MyDrive/dense_bundle_mask01.zip')

with zipfile.ZipFile(bundle) as zf:
    zf.extractall(WORK)

settings = json.loads((WORK / "dense_settings.json").read_text())
n_images = len(list((WORK / "images").iterdir()))
print(f"run_id      : {settings['run_id']}")
print(f"images      : {n_images}")
print(f"masks       : {(WORK / 'masks').is_dir()}")
print(f"max size    : {settings['max_image_size']}")
settings

## 4. Undistort

Rectifies the images and rewrites the model into the workspace layout that
patch-match stereo expects.

In [ ]:
import subprocess, time

DENSE = WORK / "dense"

def run(cmd):
    print("$", " ".join(str(c) for c in cmd))
    start = time.time()
    proc = subprocess.run([str(c) for c in cmd], capture_output=True, text=True)
    if proc.returncode != 0:
        print(proc.stdout[-3000:]); print(proc.stderr[-3000:])
        raise RuntimeError(f"failed: {cmd[1] if len(cmd) > 1 else cmd}")
    print(f"  done in {time.time() - start:.0f}s")
    return proc

run([
    "colmap", "image_undistorter",
    "--image_path", WORK / "images",
    "--input_path", WORK / "sparse",
    "--output_path", DENSE,
    "--output_type", "COLMAP",
    "--max_image_size", settings["max_image_size"],
])

## 5. Patch-match stereo (the GPU stage)

The slow step: roughly 15-40 minutes for 60-100 images on a T4.

`geom_consistency` cross-checks depth between views. It costs a second pass but
removes much of the noise that specular surfaces produce -- which on a gilded
Khon mask is the difference between a usable cloud and a spiky one.

In [ ]:
run([
    "colmap", "patch_match_stereo",
    "--workspace_path", DENSE,
    "--workspace_format", "COLMAP",
    "--PatchMatchStereo.geom_consistency",
    "true" if settings["geom_consistency"] else "false",
    "--PatchMatchStereo.window_radius", settings["window_radius"],
    "--PatchMatchStereo.num_samples", settings["num_samples"],
    "--PatchMatchStereo.max_image_size", settings["max_image_size"],
])

## 6. Fuse into a point cloud

`min_num_pixels` is the consistency requirement: a point must be seen and agreed
on by at least this many views. Raising it yields a cleaner but sparser cloud.

In [ ]:
FUSED = DENSE / "fused.ply"

run([
    "colmap", "stereo_fusion",
    "--workspace_path", DENSE,
    "--workspace_format", "COLMAP",
    "--input_type", "geometric" if settings["geom_consistency"] else "photometric",
    "--output_path", FUSED,
    "--StereoFusion.min_num_pixels", settings["fusion_min_num_pixels"],
    "--StereoFusion.max_reproj_error", settings["fusion_max_reproj_error"],
])

print(f"\nfused.ply: {FUSED.stat().st_size / 1e6:.1f} MB")

## 7. Sanity-check the result before downloading

In [ ]:
!pip -q install plyfile > /dev/null 2>&1
import numpy as np
from plyfile import PlyData

ply = PlyData.read(str(FUSED))
vertex = ply["vertex"]
points = np.stack([vertex["x"], vertex["y"], vertex["z"]], axis=1)

print(f"points        : {len(points):,}")
print(f"has colour    : {'red' in vertex.data.dtype.names}")
print(f"has normals   : {'nx' in vertex.data.dtype.names}")
print(f"bbox extent   : {(points.max(axis=0) - points.min(axis=0)).round(3)}")

assert len(points) > 1000, (
    "Very few points were fused. Usual causes: too little overlap between "
    "photos, or masks that removed nearly all of the object."
)
print("\nlooks reasonable -- download it in the next cell")

## 8. Download

Then, locally:
```
python scripts/03_dense_import.py ~/Downloads/fused.ply -s paths.run_id=<run_id>
python scripts/04_mesh.py -s paths.run_id=<run_id>
python scripts/05_texture.py -s paths.run_id=<run_id>
python scripts/06_evaluate.py -s paths.run_id=<run_id>
```

`03_dense_import.py` verifies the cloud lines up with the sparse model, so a
bundle downloaded from the wrong run is caught immediately.

In [ ]:
from google.colab import files
files.download(str(FUSED))

# The depth maps are worth keeping for the specularity study -- they show
# exactly where photometric matching failed on the gilded regions.
# !cd {DENSE} && zip -qr /content/depth_maps.zip stereo/depth_maps
# files.download('/content/depth_maps.zip')